# Módulo 6 · Clase 6 — pandas para análisis de datos

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Series y DataFrame |
| 2 | Creación, importación e inspección |
| 3 | Selección e indexación |
| 4 | Filtrado y transformación |
| 5 | Agrupación y agregación |
| 6 | Limpieza de datos (NaN, duplicados, tipos) |
| 7 | Merge, pivot y visualización básica |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 100)
print(f"pandas {pd.__version__}")

---
## Motivación

```
NumPy es perfecto para arrays numéricos homogéneos.
Pero los datos reales son:
  - Mixtos: números, texto, fechas, booleanos
  - Sucios: valores missing, duplicados, tipos incorrectos
  - Tabulares: filas (observaciones) × columnas (variables)

pandas = NumPy + índice + columnas con nombre + tipos mixtos
                + operaciones de base de datos + limpieza

Antes de entrenar cualquier modelo → sus datos pasan por pandas.
```

---
## 1. Series — el array con índice

In [ ]:
# Serie desde lista
notas = pd.Series([8.5, 7.0, 9.2, 6.1, 8.8], name="nota_final")
print("Serie simple:")
print(notas)

# Serie con índice personalizado
notas_nombres = pd.Series(
    [8.5, 7.0, 9.2, 6.1, 8.8],
    index=["Ana", "Bruno", "Carla", "Diego", "Elena"],
    name="nota_final"
)
print("\nSerie con índice de nombres:")
print(notas_nombres)

# Operaciones vectorizadas (como NumPy)
print(f"\nMedia: {notas_nombres.mean():.2f}")
print(f"Aprobados (≥5): {(notas_nombres >= 5).sum()}")
print(f"\nNota normalizada (0-1):")
print(((notas_nombres - notas_nombres.min()) / (notas_nombres.max() - notas_nombres.min())).round(3))

# Acceso por etiqueta e índice
print(f"\nNota de Carla (por nombre): {notas_nombres['Carla']}")
print(f"Primer elemento (por pos):  {notas_nombres.iloc[0]}")

---
## 2. DataFrame — la tabla de datos

In [ ]:
# Crear desde diccionario
np.random.seed(42)
n = 100

df = pd.DataFrame({
    'id':          range(1, n+1),
    'nombre':      [f"Est_{i:03d}" for i in range(1, n+1)],
    'edad':        np.random.randint(18, 35, n),
    'ciudad':      np.random.choice(['Madrid', 'Barcelona', 'Valencia', 'Sevilla'], n),
    'horas_estudio': np.random.normal(5, 1.5, n).clip(0).round(1),
    'nota_teoria': np.random.normal(6.5, 1.5, n).clip(0, 10).round(1),
    'nota_practica': np.random.normal(7.0, 1.2, n).clip(0, 10).round(1),
    'aprobado':    np.random.choice([True, False], n, p=[0.75, 0.25]),
})
df['nota_final'] = (df['nota_teoria'] * 0.4 + df['nota_practica'] * 0.6).round(1)

print("=== Primeras filas ===")
print(df.head())

print("\n=== Información del DataFrame ===")
df.info()

print("\n=== Estadísticas descriptivas ===")
print(df.describe().round(2))

print(f"\nDimensiones: {df.shape}  ({df.shape[0]} filas × {df.shape[1]} columnas)")

---
## 3. Selección e indexación

In [ ]:
# Columnas
print("Una columna:")
print(df['ciudad'].head())

print("\nVarias columnas:")
print(df[['nombre', 'nota_final', 'aprobado']].head())

# .loc  — por etiquetas (índice + nombre columna)
print("\n.loc fila 0, columnas nota_teoria y nota_practica:")
print(df.loc[0, ['nota_teoria', 'nota_practica']])

# .iloc — por posición
print("\n.iloc primeras 3 filas, columnas 0,2,4:")
print(df.iloc[:3, [0, 2, 4]])

# Slicing
print("\nFilas 5 a 8 (incluidos), todas las columnas:")
print(df.loc[5:8])

# Resumen de diferencias
print("\n=== Referencia rápida ===")
print("df['col']              → columna como Serie")
print("df[['c1','c2']]        → subconjunto de columnas")
print("df.loc[i, 'col']       → celda por etiqueta")
print("df.iloc[i, j]          → celda por posición")
print("df.loc[2:5, 'col']     → rango de filas, columna")

---
## 4. Filtrado y transformación

In [ ]:
# Filtrado booleano
aprobados = df[df['aprobado'] == True]
print(f"Aprobados: {len(aprobados)}")

# Múltiples condiciones
buenos_madrid = df[(df['ciudad'] == 'Madrid') & (df['nota_final'] >= 8)]
print(f"Nota ≥ 8 en Madrid: {len(buenos_madrid)}")

# .isin() para múltiples valores
levante = df[df['ciudad'].isin(['Barcelona', 'Valencia'])]
print(f"Barcelona o Valencia: {len(levante)}")

# .query() — sintaxis más legible
resultado = df.query("nota_final >= 8 and edad < 25")
print(f"Nota ≥ 8 y edad < 25: {len(resultado)}")

# Añadir columnas calculadas
df['rendimiento'] = df['nota_final'] / df['horas_estudio']
df['categoria']   = pd.cut(df['nota_final'],
                            bins=[0, 4.9, 6.9, 8.9, 10],
                            labels=['Suspenso', 'Aprobado', 'Notable', 'Sobresaliente'])

print("\nDistribución por categoría:")
print(df['categoria'].value_counts())

# .apply() para funciones personalizadas
def clasificar_edad(edad):
    if edad < 22: return 'junior'
    elif edad < 28: return 'mid'
    else: return 'senior'

df['grupo_edad'] = df['edad'].apply(clasificar_edad)
print("\nGrupos de edad:")
print(df['grupo_edad'].value_counts())

---
## 5. Agrupación y agregación

In [ ]:
# Estadísticas por ciudad
por_ciudad = df.groupby('ciudad')['nota_final'].agg(['mean', 'std', 'count'])
por_ciudad.columns = ['media', 'std', 'n']
por_ciudad = por_ciudad.round(3)
print("Notas por ciudad:")
print(por_ciudad)

# Múltiples columnas y estadísticos
print("\nResumen por categoría:")
resumen = df.groupby('categoria').agg(
    n=('nota_final', 'count'),
    nota_media=('nota_final', 'mean'),
    horas_media=('horas_estudio', 'mean'),
    pct_aprobados=('aprobado', 'mean')
).round(3)
print(resumen)

# Agrupación múltiple
print("\nMedia de nota por ciudad y grupo_edad:")
multi = df.groupby(['ciudad', 'grupo_edad'])['nota_final'].mean().round(2).unstack()
print(multi)

# transform: añadir estadístico de grupo como columna
df['media_ciudad'] = df.groupby('ciudad')['nota_final'].transform('mean')
df['sobre_media_ciudad'] = df['nota_final'] > df['media_ciudad']
print(f"\nEstudiantes sobre la media de su ciudad: {df['sobre_media_ciudad'].sum()}")

---
## 6. Limpieza de datos

In [ ]:
# Dataset con problemas
datos_sucios = pd.DataFrame({
    'nombre':  ['Ana', 'Bruno', 'Carla', 'Diego', 'Ana', 'Elena', None, 'Fran'],
    'edad':    [25, None, 30, 22, 25, 28, 31, -5],
    'nota':    [8.5, 7.0, None, 6.1, 8.5, None, 9.2, 7.8],
    'ciudad':  ['Madrid', 'madrid', 'BARCELONA', 'Valencia', 'Madrid', 'Sevilla', 'Barcelona', 'Madrid'],
    'email':   ['ana@mail.com', 'bruno@', 'carla@mail.com', 'diego@mail.com',
                'ana@mail.com', 'elena@mail.com', None, 'fran@mail.com'],
})

print("Dataset con problemas:")
print(datos_sucios)

# 1. Missing values
print("\n=== Missing values ===")
print(datos_sucios.isnull().sum())
print(f"Total: {datos_sucios.isnull().sum().sum()}")

# Rellenar / eliminar
df_clean = datos_sucios.copy()
df_clean['nota'].fillna(df_clean['nota'].median(), inplace=True)
df_clean['edad'].fillna(df_clean['edad'].median(), inplace=True)
df_clean.dropna(subset=['nombre'], inplace=True)  # eliminar filas sin nombre

# 2. Duplicados
print("\n=== Duplicados ===")
print(f"Filas duplicadas: {df_clean.duplicated().sum()}")
df_clean = df_clean.drop_duplicates()

# 3. Normalizar texto
df_clean['ciudad'] = df_clean['ciudad'].str.strip().str.title()
df_clean['nombre'] = df_clean['nombre'].str.strip().str.title()

# 4. Valores imposibles
print(f"\nEdades negativas: {(df_clean['edad'] < 0).sum()}")
df_clean = df_clean[df_clean['edad'] > 0]

# 5. Tipos de datos
df_clean['edad'] = df_clean['edad'].astype(int)

print("\nDataset limpio:")
print(df_clean.reset_index(drop=True))

---
## 7. Merge, pivot y visualización

In [ ]:
# Merge (equivalente al JOIN de SQL)
estudiantes = pd.DataFrame({
    'id_est': [1, 2, 3, 4, 5],
    'nombre': ['Ana', 'Bruno', 'Carla', 'Diego', 'Elena'],
    'id_ciudad': [10, 10, 20, 30, 20]
})
ciudades = pd.DataFrame({
    'id_ciudad': [10, 20, 30],
    'ciudad': ['Madrid', 'Barcelona', 'Valencia'],
    'region': ['Centro', 'Cataluña', 'Comunidad Valenciana']
})

combinado = pd.merge(estudiantes, ciudades, on='id_ciudad', how='left')
print("Merge (left join) estudiantes + ciudades:")
print(combinado)

# Pivot table
print("\nTabla pivot: media de nota por ciudad y categoría:")
pivot = df.pivot_table(
    values='nota_final',
    index='ciudad',
    columns='grupo_edad',
    aggfunc='mean'
).round(2)
print(pivot)

# Visualización con pandas
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Histograma
df['nota_final'].hist(bins=20, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de nota final')
axes[0].set_xlabel('Nota')

# Barras por ciudad
df.groupby('ciudad')['nota_final'].mean().sort_values().plot(
    kind='barh', ax=axes[1], color='coral'
)
axes[1].set_title('Media nota por ciudad')
axes[1].set_xlabel('Nota media')

# Boxplot por categoría
df.boxplot(column='horas_estudio', by='categoria', ax=axes[2],
           flierprops=dict(marker='o', ms=4, alpha=0.5))
axes[2].set_title('Horas de estudio por categoría')
plt.suptitle('')

plt.tight_layout(); plt.show()

---
## Práctica

### Ejercicio 1 — Exploración inicial

**Enunciado:** Crea este dataset de ventas de una tienda y explóralo:

```python
np.random.seed(1)
n = 200
ventas = pd.DataFrame({
    'fecha':     pd.date_range('2024-01-01', periods=n, freq='D'),
    'producto':  np.random.choice(['laptop','tablet','phone','smartwatch'], n),
    'cantidad':  np.random.randint(1, 20, n),
    'precio':    np.random.choice([299, 499, 799, 1299], n),
    'descuento': np.random.uniform(0, 0.3, n).round(2),
    'vendedor':  np.random.choice(['Alice','Bob','Carol','Dave'], n),
})
ventas['total'] = (ventas['cantidad'] * ventas['precio'] * (1 - ventas['descuento'])).round(2)
```

1. `shape`, `dtypes`, `info()`, `describe()`
2. ¿Cuántas ventas hay de cada producto?
3. ¿Qué vendedor tiene la mayor facturación total?
4. Filtra ventas con descuento > 20% y total > 5000€
5. Añade una columna `mes` extrayendo el mes de `fecha`

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import pandas as pd
import numpy as np

np.random.seed(1)
n = 200
ventas = pd.DataFrame({
    'fecha':     pd.date_range('2024-01-01', periods=n, freq='D'),
    'producto':  np.random.choice(['laptop','tablet','phone','smartwatch'], n),
    'cantidad':  np.random.randint(1, 20, n),
    'precio':    np.random.choice([299, 499, 799, 1299], n),
    'descuento': np.random.uniform(0, 0.3, n).round(2),
    'vendedor':  np.random.choice(['Alice','Bob','Carol','Dave'], n),
})
ventas['total'] = (ventas['cantidad'] * ventas['precio'] * (1 - ventas['descuento'])).round(2)

# 1.
print(f"Shape: {ventas.shape}")
print(ventas.dtypes)
ventas.info()
print(ventas.describe().round(2))

# 2.
print("\nVentas por producto:")
print(ventas['producto'].value_counts())

# 3.
facturacion = ventas.groupby('vendedor')['total'].sum().sort_values(ascending=False)
print(f"\nFacturación por vendedor:")
print(facturacion)
print(f"\nMejor vendedor: {facturacion.index[0]} ({facturacion.iloc[0]:,.2f}€)")

# 4.
filtro = ventas[(ventas['descuento'] > 0.20) & (ventas['total'] > 5000)]
print(f"\nVentas con descuento >20% y total >5000€: {len(filtro)}")
print(filtro[['producto', 'cantidad', 'precio', 'descuento', 'total']].head())

# 5.
ventas['mes'] = ventas['fecha'].dt.month
print(f"\nVentas por mes:")
print(ventas.groupby('mes')['total'].sum().round(2))
```
</details>

---
### Ejercicio 2 — Limpieza completa

**Enunciado:** Limpia este dataset con múltiples problemas:

```python
datos = pd.DataFrame({
    'id':     [1, 2, 3, 4, 5, 6, 7, 2],          # duplicado
    'nombre': ['Ana', 'Bruno', None, 'Diego', 'ELENA', 'fran', 'Gema', 'Bruno'],
    'edad':   [25, 28, 999, 22, 30, -3, None, 28],  # edades imposibles
    'salario':[45000, None, 38000, 52000, 61000, 29000, None, 47000],
    'dpto':   ['IT', 'IT', 'HR', 'hr', 'Finance', 'FINANCE', 'IT', 'IT'],
    'activo': ['Si', 'No', 'si', 'SI', 'no', 'NO', 'Si', 'No'],
})
```

1. Detecta todos los problemas
2. Elimina duplicados (mantén el primero)
3. Limpia texto (nombre, dpto, activo → True/False)
4. Corrige edades imposibles → NaN
5. Imputa salarios con la mediana del departamento

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import pandas as pd
import numpy as np

datos = pd.DataFrame({
    'id':     [1, 2, 3, 4, 5, 6, 7, 2],
    'nombre': ['Ana', 'Bruno', None, 'Diego', 'ELENA', 'fran', 'Gema', 'Bruno'],
    'edad':   [25, 28, 999, 22, 30, -3, None, 28],
    'salario':[45000, None, 38000, 52000, 61000, 29000, None, 47000],
    'dpto':   ['IT', 'IT', 'HR', 'hr', 'Finance', 'FINANCE', 'IT', 'IT'],
    'activo': ['Si', 'No', 'si', 'SI', 'no', 'NO', 'Si', 'No'],
})

df = datos.copy()

# 1. Diagnóstico
print("Problemas detectados:")
print(f"  Filas duplicadas: {df.duplicated().sum()}")
print(f"  Nulls:\n{df.isnull().sum()}")
print(f"  Edades imposibles (>120 o <0): {((df['edad']>120)|(df['edad']<0)).sum()}")

# 2. Eliminar duplicados
df = df.drop_duplicates(subset='id', keep='first')

# 3. Limpiar texto
df['nombre'] = df['nombre'].str.strip().str.title()
df['dpto']   = df['dpto'].str.strip().str.upper()
df['activo'] = df['activo'].str.strip().str.lower().map({'si': True, 'no': False})

# 4. Edades imposibles → NaN
df['edad'] = df['edad'].where((df['edad'] >= 16) & (df['edad'] <= 80), other=np.nan)

# 5. Imputar salario con mediana del dpto
mediana_dpto = df.groupby('dpto')['salario'].transform('median')
df['salario'] = df['salario'].fillna(mediana_dpto)

print("\nDataset limpio:")
print(df.to_string(index=False))
print(f"\nNulls restantes: {df.isnull().sum().sum()}")
```
</details>

---
### Ejercicio 3 — Análisis de agrupaciones

**Enunciado:** Usando el DataFrame `df` generado al inicio de la clase:

1. Media, máximo y mínimo de `nota_final` por `ciudad` y `grupo_edad`
2. ¿Qué ciudad tiene mayor tasa de aprobados?
3. Crea una columna `ranking` con el rango de nota dentro de cada ciudad (1 = mejor nota)
4. Top 3 estudiantes por ciudad según nota final

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
# 1.
stats_ciudad_edad = df.groupby(['ciudad', 'grupo_edad'])['nota_final'].agg(['mean','max','min']).round(2)
print("Nota final por ciudad y grupo de edad:")
print(stats_ciudad_edad)

# 2.
tasa_aprobados = df.groupby('ciudad')['aprobado'].mean().sort_values(ascending=False)
print(f"\nTasa de aprobados por ciudad:")
print(tasa_aprobados.round(3))
print(f"\nCiudad con mayor tasa: {tasa_aprobados.index[0]}")

# 3. Ranking dentro de ciudad
df['ranking_ciudad'] = df.groupby('ciudad')['nota_final'].rank(ascending=False, method='min').astype(int)
print("\nPrimeros registros con ranking:")
print(df[['nombre', 'ciudad', 'nota_final', 'ranking_ciudad']].head(10))

# 4. Top 3 por ciudad
print("\nTop 3 por ciudad:")
top3 = df[df['ranking_ciudad'] <= 3].sort_values(['ciudad', 'ranking_ciudad'])
print(top3[['ciudad', 'ranking_ciudad', 'nombre', 'nota_final']].to_string(index=False))
```
</details>

---
### Ejercicio 4 — Merge y pivot

**Enunciado:** Combina estos dos DataFrames y crea análisis cruzados:

```python
np.random.seed(10)
cursos_df = pd.DataFrame({
    'id_curso': range(1, 6),
    'nombre_curso': ['Python', 'ML', 'Deep Learning', 'SQL', 'Cloud'],
    'nivel': ['basico', 'intermedio', 'avanzado', 'basico', 'intermedio'],
    'duracion_h': [20, 40, 60, 15, 30]
})
inscripciones_df = pd.DataFrame({
    'id_est': np.random.randint(1, 21, 80),
    'id_curso': np.random.randint(1, 6, 80),
    'nota': np.random.normal(6.5, 1.5, 80).clip(0, 10).round(1),
    'completado': np.random.choice([True, False], 80, p=[0.8, 0.2])
})
```

1. Haz merge de inscripciones con cursos
2. Nota media por curso (ordenada)
3. Tabla pivot: nota media × nivel × completado
4. ¿Qué nivel tiene mayor tasa de completado?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import pandas as pd
import numpy as np

np.random.seed(10)
cursos_df = pd.DataFrame({
    'id_curso': range(1, 6),
    'nombre_curso': ['Python', 'ML', 'Deep Learning', 'SQL', 'Cloud'],
    'nivel': ['basico', 'intermedio', 'avanzado', 'basico', 'intermedio'],
    'duracion_h': [20, 40, 60, 15, 30]
})
inscripciones_df = pd.DataFrame({
    'id_est': np.random.randint(1, 21, 80),
    'id_curso': np.random.randint(1, 6, 80),
    'nota': np.random.normal(6.5, 1.5, 80).clip(0, 10).round(1),
    'completado': np.random.choice([True, False], 80, p=[0.8, 0.2])
})

# 1. Merge
df_m = pd.merge(inscripciones_df, cursos_df, on='id_curso', how='left')
print("Merged (primeras filas):")
print(df_m.head())

# 2. Nota media por curso
print("\nNota media por curso:")
nota_curso = df_m.groupby('nombre_curso')['nota'].mean().sort_values(ascending=False).round(3)
print(nota_curso)

# 3. Pivot: nivel × completado
print("\nNota media por nivel y completado:")
pivot = df_m.pivot_table(values='nota', index='nivel', columns='completado', aggfunc='mean').round(3)
pivot.columns = ['No completado', 'Completado']
print(pivot)

# 4. Tasa de completado por nivel
tasa = df_m.groupby('nivel')['completado'].mean().sort_values(ascending=False)
print(f"\nTasa de completado por nivel:")
print(tasa.round(3))
print(f"Nivel con mayor tasa: {tasa.index[0]}")
```
</details>

---
### Ejercicio 5 (reto) — Pipeline de preprocesamiento

**Enunciado:** Implementa una función `preprocesar(df)` que recibe un DataFrame crudo y devuelve uno limpio y enriquecido, aplicando en orden:

1. Eliminar duplicados
2. Normalizar texto (columnas tipo object)
3. Corregir outliers numéricos (reemplaza por NaN los valores fuera de Q1-3×IQR, Q3+3×IQR)
4. Imputar NaN numéricos con la mediana de su columna
5. Añadir columna `calidad_dato` = porcentaje de campos no nulos que tenía el registro original
6. Devuelve (df_limpio, reporte_dict) con estadísticas del proceso

Pruébala con el dataset de ventas del ejercicio 1.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import pandas as pd
import numpy as np

def preprocesar(df_raw):
    df = df_raw.copy()
    reporte = {}

    # Calidad original (antes de cualquier cambio)
    calidad_original = df.notna().mean(axis=1) * 100

    # 1. Duplicados
    antes = len(df)
    df = df.drop_duplicates()
    reporte['filas_eliminadas_dup'] = antes - len(df)

    # 2. Normalizar texto
    cols_obj = df.select_dtypes(include='object').columns
    for col in cols_obj:
        df[col] = df[col].str.strip().str.title()
    reporte['cols_texto_normalizadas'] = list(cols_obj)

    # 3. Outliers → NaN (usando 3×IQR, más conservador que 1.5×IQR)
    cols_num = df.select_dtypes(include=np.number).columns
    outliers_total = 0
    for col in cols_num:
        Q1, Q3 = df[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        mask = (df[col] < Q1 - 3*IQR) | (df[col] > Q3 + 3*IQR)
        n_out = mask.sum()
        outliers_total += n_out
        df.loc[mask, col] = np.nan
    reporte['outliers_convertidos_nan'] = outliers_total

    # 4. Imputar NaN numéricos
    nan_antes = df[cols_num].isnull().sum().sum()
    for col in cols_num:
        df[col].fillna(df[col].median(), inplace=True)
    reporte['nan_imputados'] = nan_antes

    # 5. Calidad del dato
    df['calidad_dato'] = calidad_original.reindex(df.index).fillna(0).round(1)

    reporte['filas_finales']   = len(df)
    reporte['columnas_finales'] = len(df.columns)

    return df, reporte

# Prueba con ventas
np.random.seed(1)
n = 200
ventas = pd.DataFrame({
    'fecha':     pd.date_range('2024-01-01', periods=n, freq='D'),
    'producto':  np.random.choice(['laptop','tablet','phone','smartwatch'], n),
    'cantidad':  np.random.randint(1, 20, n),
    'precio':    np.random.choice([299, 499, 799, 1299], n),
    'descuento': np.random.uniform(0, 0.3, n).round(2),
    'vendedor':  np.random.choice(['Alice','Bob','Carol','Dave'], n),
})
ventas['total'] = (ventas['cantidad'] * ventas['precio'] * (1 - ventas['descuento'])).round(2)
# Añadir problemas artificiales
ventas.loc[0, 'cantidad'] = None
ventas.loc[1, 'precio']   = 99999  # outlier
ventas = pd.concat([ventas, ventas.iloc[[5]]])  # duplicado

df_limpio, reporte = preprocesar(ventas)
print("Reporte de preprocesamiento:")
for k, v in reporte.items():
    print(f"  {k}: {v}")
print(f"\nCalidad media de datos: {df_limpio['calidad_dato'].mean():.1f}%")
```
</details>

---
## Resumen

```python
import pandas as pd

# Crear y explorar
df = pd.DataFrame(diccionario)
df.info();  df.describe();  df.head()

# Seleccionar
df['col'];  df[['c1','c2']]
df.loc[fila, col];  df.iloc[i, j]

# Filtrar
df[df['col'] > val]
df.query("col > val and otro == 'x'")

# Transformar
df['nueva'] = df['a'] + df['b']
df['cat']   = pd.cut(df['num'], bins=[...])
df['res']   = df['col'].apply(mi_funcion)

# Agrupar
df.groupby('col')['val'].agg(['mean','std','count'])
df.groupby('col')['val'].transform('mean')  # devuelve misma longitud

# Limpiar
df.isnull().sum();  df.drop_duplicates()
df['col'].fillna(df['col'].median())
df.dropna(subset=['col_critica'])

# Combinar
pd.merge(df1, df2, on='key', how='left')
df.pivot_table(values='v', index='i', columns='c', aggfunc='mean')
```

## ¿Qué viene en la próxima clase?
En la **Clase 7** cerramos el módulo con un **análisis exploratorio completo** (EDA): cargamos un dataset real, aplicamos todo lo aprendido en el módulo y extraemos conclusiones accionables.